In [144]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

In [145]:
df = pd.read_csv('loan.csv')
df.head()

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y


In [146]:
df.isnull().sum()

Loan_ID               0
Gender               13
Married               3
Dependents           15
Education             0
Self_Employed        32
ApplicantIncome       0
CoapplicantIncome     0
LoanAmount           22
Loan_Amount_Term     14
Credit_History       50
Property_Area         0
Loan_Status           0
dtype: int64

In [147]:
df.fillna(df.select_dtypes(include='number').mean(), inplace=True)

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.0,146.412162,360.0,1.0,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.000000,360.0,1.0,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.000000,360.0,1.0,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.000000,360.0,1.0,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.000000,360.0,1.0,Urban,Y
...,...,...,...,...,...,...,...,...,...,...,...,...,...
609,LP002978,Female,No,0,Graduate,No,2900,0.0,71.000000,360.0,1.0,Rural,Y
610,LP002979,Male,Yes,3+,Graduate,No,4106,0.0,40.000000,180.0,1.0,Rural,Y
611,LP002983,Male,Yes,1,Graduate,No,8072,240.0,253.000000,360.0,1.0,Urban,Y
612,LP002984,Male,Yes,2,Graduate,No,7583,0.0,187.000000,360.0,1.0,Urban,Y


In [148]:
df.isnull().sum()

Loan_ID               0
Gender               13
Married               3
Dependents           15
Education             0
Self_Employed        32
ApplicantIncome       0
CoapplicantIncome     0
LoanAmount            0
Loan_Amount_Term      0
Credit_History        0
Property_Area         0
Loan_Status           0
dtype: int64

In [149]:
## For categorical columns, we can fill missing values with the mode (most frequent value)
df.fillna({
    "Gender": df["Gender"].mode()[0],
    "Married": df["Married"].mode()[0],
    "Dependents": df["Dependents"].mode()[0],
    "Self_Employed": df["Self_Employed"].mode()[0]
}, inplace=True)

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.0,146.412162,360.0,1.0,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.000000,360.0,1.0,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.000000,360.0,1.0,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.000000,360.0,1.0,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.000000,360.0,1.0,Urban,Y
...,...,...,...,...,...,...,...,...,...,...,...,...,...
609,LP002978,Female,No,0,Graduate,No,2900,0.0,71.000000,360.0,1.0,Rural,Y
610,LP002979,Male,Yes,3+,Graduate,No,4106,0.0,40.000000,180.0,1.0,Rural,Y
611,LP002983,Male,Yes,1,Graduate,No,8072,240.0,253.000000,360.0,1.0,Urban,Y
612,LP002984,Male,Yes,2,Graduate,No,7583,0.0,187.000000,360.0,1.0,Urban,Y


In [150]:
df.isnull().sum()

Loan_ID              0
Gender               0
Married              0
Dependents           0
Education            0
Self_Employed        0
ApplicantIncome      0
CoapplicantIncome    0
LoanAmount           0
Loan_Amount_Term     0
Credit_History       0
Property_Area        0
Loan_Status          0
dtype: int64

In [151]:
df.shape

(614, 13)

In [152]:
df.drop(columns=['Loan_ID', 'CoapplicantIncome'], inplace= True)
df.head()

,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,Male,No,0,Graduate,No,5849,146.412162,360.0,1.0,Urban,Y
1,Male,Yes,1,Graduate,No,4583,128.000000,360.0,1.0,Rural,N
2,Male,Yes,0,Graduate,Yes,3000,66.000000,360.0,1.0,Urban,Y
3,Male,Yes,0,Not Graduate,No,2583,120.000000,360.0,1.0,Urban,Y
4,Male,No,0,Graduate,No,6000,141.000000,360.0,1.0,Urban,Y


### train test split

In [153]:
X_train, X_test, y_train, y_test = train_test_split(
    df.iloc[:, :-1],   # all columns except last
    df.iloc[:, -1],    # only last column
    test_size=0.2
)

In [154]:
encoder = LabelEncoder()

cols_to_encode = [
    "Gender",
    "Married",
    "Dependents",
    "Education",
    "Self_Employed",
    "Credit_History",
    "Property_Area"
]

for col in cols_to_encode:
    X_train[col] = encoder.fit_transform(X_train[col])
    X_test[col] = encoder.transform(X_test[col])

scaler = StandardScaler()

cols_to_scale = ["ApplicantIncome", "LoanAmount", "Loan_Amount_Term"]

X_train[cols_to_scale] = scaler.fit_transform(X_train[cols_to_scale])
X_test[cols_to_scale] = scaler.transform(X_test[cols_to_scale])

X_train

,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area
8,1,1,2,0,0,-0.221938,0.235734,0.277634,2,2
488,1,1,2,0,1,-0.128544,0.142147,0.277634,2,1
176,1,1,2,0,0,-0.539345,-0.548055,0.277634,2,0
192,1,1,0,1,0,0.106153,0.142147,0.277634,2,2
570,1,1,1,0,0,-0.317273,0.446304,0.277634,2,2
...,...,...,...,...,...,...,...,...,...,...
25,1,1,0,0,1,0.677034,0.504796,0.277634,2,1
525,1,1,2,0,1,1.962203,2.949748,0.277634,2,0
246,1,1,2,0,0,0.700180,-0.419373,0.277634,2,2
587,0,0,0,1,0,-0.519922,-0.910703,0.277634,2,1


In [155]:
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

y_train

array([1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 1,
       0, 1, 1, 0, 0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 0, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1,
       0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 0, 1,
       1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 1,
       1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1,
       1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1,
       1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1,
       0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 0, 1,
       1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1,
       1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0,
       1, 1, 1, 1, 1, 1, 0, 1, 0, 0, 1, 1, 1, 0, 1,

In [156]:
X_train_tensor = torch.from_numpy(X_train.values).float()
X_test_tensor = torch.from_numpy(X_test.values).float()

y_train_tensor = torch.from_numpy(y_train).float()
y_test_tensor = torch.from_numpy(y_test).float()

In [157]:
X_train_tensor.shape

torch.Size([491, 10])

In [158]:
y_train_tensor.shape

torch.Size([491])

In [159]:
from torch.utils.data import Dataset, DataLoader

class CustomDataset(Dataset):

  def __init__(self, features, labels):

    self.features = features
    self.labels = labels

  def __len__(self):

    return self.features.shape[0]

  def __getitem__(self, index):

    return self.features[index], self.labels[index]

In [160]:
train_dataset = CustomDataset(X_train_tensor, y_train_tensor)
test_dataset = CustomDataset(X_test_tensor, y_test_tensor)

In [161]:
train_dataset[10]

(tensor([ 1.0000,  1.0000,  2.0000,  0.0000,  0.0000,  0.0715, -0.3258,  0.2776,
          2.0000,  0.0000]),
 tensor(1.))

In [162]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

### Defining the model

In [163]:
class MySimpleNN(nn.Module):

  def __init__(self, num_features):
    super().__init__()
    self.network = nn.Sequential(
        nn.Linear(in_features=num_features, out_features=128),
        nn.ReLU(),
        nn.Linear(128, 64),
        nn.ReLU(),
        nn.Linear(64, 32),
        nn.ReLU(),
        nn.Linear(32, 1),
        nn.Sigmoid()
    )

  def forward(self, features):
    y_pred = self.network(features)
    return y_pred



In [164]:
# set learning rate and epochs
epochs = 100
learning_rate = 0.1

In [165]:
#buitin loss function
loss_fn = nn.BCELoss()

In [166]:
### model initialization
model = MySimpleNN(num_features=X_train_tensor.shape[1])

#builtin optimizer
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

#### training loop

In [167]:
for epoch in range(epochs):
    
    for batch_features, batch_labels in train_loader:
        
        # forward pass
        y_pred = model(batch_features)

        # loss calculation
        loss = loss_fn(y_pred.squeeze(), batch_labels)

        # zero gradients
        optimizer.zero_grad()

        # backward pass
        loss.backward()

        # update weights
        optimizer.step()

    # print loss after each epoch
    print(f'Epoch: {epoch + 1}, Loss: {loss.item()}')

Epoch: 1, Loss: 0.6133629083633423
Epoch: 2, Loss: 0.5110943913459778
Epoch: 3, Loss: 0.5293300747871399
Epoch: 4, Loss: 0.4920189678668976
Epoch: 5, Loss: 0.46967455744743347
Epoch: 6, Loss: 0.5520387887954712
Epoch: 7, Loss: 0.32660356163978577
Epoch: 8, Loss: 0.47051355242729187
Epoch: 9, Loss: 0.5482138991355896
Epoch: 10, Loss: 0.38664111495018005
Epoch: 11, Loss: 0.45398205518722534
Epoch: 12, Loss: 0.4022999703884125
Epoch: 13, Loss: 0.3984615206718445
Epoch: 14, Loss: 0.31688427925109863
Epoch: 15, Loss: 0.27253973484039307
Epoch: 16, Loss: 0.43228432536125183
Epoch: 17, Loss: 0.5092190504074097
Epoch: 18, Loss: 0.3858968913555145
Epoch: 19, Loss: 0.48645564913749695
Epoch: 20, Loss: 0.39658859372138977
Epoch: 21, Loss: 0.488413542509079
Epoch: 22, Loss: 0.3899105191230774
Epoch: 23, Loss: 0.25756731629371643
Epoch: 24, Loss: 0.43574705719947815
Epoch: 25, Loss: 0.21932129561901093
Epoch: 26, Loss: 0.3287009596824646
Epoch: 27, Loss: 0.49502497911453247
Epoch: 28, Loss: 0.33097

### Evaluation

In [168]:
#model evaluation using the test loader
model.eval()  # Set the model to evaluation mode
accuracy_list = []
predictions_list = []

with torch.inference_mode():
    for batch_features, batch_labels in test_loader:
        y_pred = model(batch_features)
        predicted_labels = (y_pred > 0.5).float()
        predictions_list.extend(predicted_labels.view(-1).tolist())
        accuracy = (predicted_labels.view(-1) == batch_labels).float().mean()
        accuracy_list.append(accuracy.item())

#calculate overall accuracy
overall_predictions = np.array(predictions_list)
print(f'Predictions: {overall_predictions}')
overall_accuracy = sum(accuracy_list) / len(accuracy_list)
print(f'Overall Accuracy: {overall_accuracy * 100:.2f}%')

Predictions: [0. 0. 0. 1. 1. 0. 1. 1. 1. 1. 0. 1. 0. 1. 1. 1. 1. 0. 1. 1. 1. 0. 1. 0.
 0. 1. 1. 1. 0. 1. 0. 1. 0. 0. 1. 1. 1. 1. 0. 0. 1. 1. 0. 1. 1. 1. 1. 1.
 1. 0. 0. 1. 0. 1. 1. 1. 1. 0. 1. 0. 0. 1. 1. 1. 0. 0. 1. 1. 1. 1. 1. 0.
 1. 1. 0. 1. 1. 1. 1. 0. 1. 1. 1. 1. 1. 0. 1. 0. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 0. 1. 1. 1. 0. 0. 1. 1. 1. 0. 1. 1. 1. 0. 1. 0. 1. 0. 1. 0. 1. 1. 0.
 1. 0. 1.]
Overall Accuracy: 68.66%


## with user input as list 

In [181]:
# ───────────────────────────────────────────────
# Predict for a new applicant
# ───────────────────────────────────────────────

# LabelEncoder mapping (from your notebook):
# Gender        → Male=1,     Female=0
# Married       → Yes=1,      No=0
# Dependents    → 0=0, 1=1, 2=2, 3+=3
# Education     → Graduate=0, Not Graduate=1
# Self_Employed → No=0,       Yes=1
# Credit_History→ 1.0=1,      0.0=0
# Property_Area → Rural=0,    Semiurban=1, Urban=2

new_applicant = np.array([[
    1,      # Gender
    1,      # Married
    0,      # Dependents
    0,      # Education
    0,      # Self_Employed
    8000,   # ApplicantIncome   ← will be scaled
    100,    # LoanAmount        ← will be scaled
    360,    # Loan_Amount_Term  ← will be scaled
    1,      # Credit_History
    2       # Property_Area
]], dtype=np.float32)

# ✅ correct indices now — [5, 6, 7]
new_applicant[:, [5, 6, 7]] = scaler.transform(new_applicant[:, [5, 6, 7]])

input_tensor = torch.from_numpy(new_applicant)



c:\Users\gamin\OneDrive\Desktop\DL_Projects\env\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [182]:
# Predict
model.eval()
with torch.inference_mode():
    output = model(input_tensor)
    probability = output.item()
    prediction = 1 if probability >= 0.5 else 0

print(f'Approval Probability : {probability:.4f}')
print(f'Loan Status          : {"✅ APPROVED" if prediction == 1 else "❌ REJECTED"}')

Approval Probability : 0.6540
Loan Status          : ✅ APPROVED
